In [1]:
import sys
import os

sys.path.append('/home/paule/open_mc_projects/windowed_multipole/02_working_notebook_vectfit')
sys.path.append('/home/paule/open_mc_projects/xs_lib/endfb-vii.1-hdf5/neutron')
sys.path.append('/home/paule/open_mc_projects/MC-1D_DT')

""

from src.geometry_classes import Geometry, Material, Neutron, Source, _BatchSource
import src.geometry_classes as geom
import src.performance_classes as perf
import src.tally_classes as tally
import src.vectfit as vf
import src.export_simulation_v3 as xpsim
import src.export_print_csv as xpcsv
import src.reconr_v2 as reconr

import openmc

openmc.config['cross_sections'] = "/home/paule/open_mc_projects/xs_lib/endfb-vii.1-hdf5/cross_sections.xml"





['/home/paule/anaconda3/envs/vectfit39/lib/python39.zip', '/home/paule/anaconda3/envs/vectfit39/lib/python3.9', '/home/paule/anaconda3/envs/vectfit39/lib/python3.9/lib-dynload', '', '/home/paule/anaconda3/envs/vectfit39/lib/python3.9/site-packages', '/home/paule/open_mc_projects/windowed_multipole/02_working_notebook_vectfit', '/home/paule/open_mc_projects/xs_lib/endfb-vii.1-hdf5/neutron', '/home/paule/open_mc_projects/MC-1D_DT', '/home/paule/open_mc_projects/windowed_multipole/02_working_notebook_vectfit']


In [2]:
# One slab of U238 — simple test case


# ============================================================================
# Uranium atom density
# ============================================================================
# Uranium metal density ~ 19.1 g/cm³,
# N = rho * Na / A  (atoms/cm³)
rho_U   = 19.1               # g/cm³
NA      = 6.02214076e23      # atoms/mol

x_U235 = 0.00                # 0% enrichment
x_U238 = 1.0 - x_U235

M_U8   = 238.05078826        # g/mol
M_U5   = 235.0439299         # g/mol
N_total_U8 = rho_U * NA / (x_U238 * M_U8 + x_U235 * M_U5)
N_total_U5 = rho_U * NA / (x_U235 * M_U5 + x_U238 * M_U8)

N_U235 = 1.0 * N_total_U5   # 0%  enrichment
N_U238 = 1.0 * N_total_U8   # 100%

slab1 = Material(
    name     = "cell 1",
    nuclides = [('U238', N_U238)],
    T        = 293.6,    # K (~20 °C)
)

slab2 = Material(
    name     = "cell 2",
    nuclides = [('U238', N_U238)],
    T        = 2000,     # K
)


('U238', 4.831863374901811e+22)
('U238', 4.831863374901811e+22)


In [3]:



# ============================================================================
# Geometry  (two slabs)
# ============================================================================
geom = Geometry(majorant_log=True, verbose=False)
geom.xs_dir = '/home/paule/open_mc_projects/xs_lib/endfb-vii.1-hdf5/wmp'

geom.add_material(slab1)
geom.add_material(slab2)
geom.boundaries     = [0.0, 2.0, 15.0]  # cm
geom.material_array = [slab1, slab2]

# Flux tally: 5 energy groups
energy_bins = [10.0, 50.0, 600.0, 1e4, 1e6, 2e7]  # eV
geom.attach_flux_tally(energy_bins)

# Verification tally: same energy bins, surface at slab1/slab2 interface
geom.attach_verification_tally(
    energy_bins = energy_bins,
    surface_xs  = [2.0],        # slab1/slab2 interface
)

# Majorant and XS method
geom.maj_mat_method = "simple"  # majorant method: "maj_mat" 
geom.set_maj_xs_method(method="vectfit", T_array=[0, 293, 500, 1000, 1500, 1900, 2000], xs_maj_file_dir='src/vectfit_data')  # majorant XS method: "vectfit" 
geom.set_access_method("reconr", last_energy= 600, err_lim=0.001, err_max=0.01)  # access method: "reconr" or "simple"
geom.set_mode("analysis", filename='validation/xs_generation/statepoint.200.h5')


# ============================================================================
# Source  (monoenergetic, at x=0, pointing right)
# ============================================================================
N_NEUTRONS  = 10000   # neutrons per batch
N_BATCHES   = 10    # number of independent batches (100 neutrons each)

src = Source(
    neutron_nbr  = N_NEUTRONS,
    energy_range = [500, 150],
    energy_dist  = "mono",
    position     = [0.0, 0.0, 0.0],
    direction    = [1.0, 0.0, 0.0],
)




U238
U238
15.0
Building majorant XS grid for RECONR access method with settings:
xs_method:      vectfit
maj_mat_method: simple
materials:      ['cell 1', 'cell 2']
4309
600
 Evaluating the majorant cross section with RECONR stacking algorithm
err_lim = 0.001, err_max = 0.01, err_int = 5e-08
last energy to add = 599.5985648781146 eV
number of windows = 4309
done
Second pass done — 3561 points
No inversions found in energy grid before dedup
Deduplication removed 0 points
Building O(1) Window Pointers...
Window pointer table: 747 windows
Final grid size:      3561 points
Window Pointers: [0, 51, 66, 73, 78, 81, 83, 85, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 141, 154, 158, 163, 164, 167, 168, 170, 172, 174, 177, 181, 185, 192, 200, 215, 258, 292, 317, 341, 388, 406, 414, 421, 425

In [4]:
import numpy as np
print(geom.reconr_e_grid[2661])
print(geom.reconr_e_grid[2662])
print(geom.reconr_maj_xs_grid[2661])
print(geom.reconr_maj_xs_grid[2662])
diffs = np.diff(geom.reconr_e_grid)
print("Energy grid differences around index 2661:", diffs[2659:2663])
print("Energy grid values at windows pointer boundaries:",
      [geom.reconr_e_grid[geom.reconr_window_pointers[i]] for i in range(631, 634)])
print(geom.reconr_window_pointers[633])

#0.6060417549796722 0.5486206785877346 429.0047406 429.0047407133517


429.0047406
429.0047407133517
0.6060417549796722
0.5486206785877346
Energy grid differences around index 2661: [3.00000011e-07 1.99999988e-07 1.13351689e-07 1.36063099e+00]
Energy grid values at windows pointer boundaries: [429.0047407133517, 430.36537170806434, 431.7281569758163]
2669


In [7]:
batch_stats = geom.run_batch(src, n_batches=N_BATCHES)
print(f"mode: {geom.mode}")
print(f"majorant XS method: {geom.maj_xs_method}")
print(f"majorant material method: {geom.maj_mat_method}")
print(f"access method: {geom.access_method}")
print(geom.summary())

xpcsv.export_cross_batch_stats(batch_stats, geom=geom, print_to_console= False, save_csv= True, output_dir=f'cross_batch_stats_{geom.maj_xs_method}_{geom.maj_mat_method}_reconr_v2_new.csv')



Running source with following setttings:
  Mode: analysis
  Majorant XS method: vectfit
  Material majorant method: simple
  Access method: reconr
  Batch   1/10  (10000 neutrons)  wall=30.490s
Running source with following setttings:
  Mode: analysis
  Majorant XS method: vectfit
  Material majorant method: simple
  Access method: reconr
  Batch   2/10  (10000 neutrons)  wall=29.690s
Running source with following setttings:
  Mode: analysis
  Majorant XS method: vectfit
  Material majorant method: simple
  Access method: reconr
  Batch   3/10  (10000 neutrons)  wall=29.517s
Running source with following setttings:
  Mode: analysis
  Majorant XS method: vectfit
  Material majorant method: simple
  Access method: reconr
  Batch   4/10  (10000 neutrons)  wall=29.599s
Running source with following setttings:
  Mode: analysis
  Majorant XS method: vectfit
  Material majorant method: simple
  Access method: reconr
  Batch   5/10  (10000 neutrons)  wall=29.816s
Running source with following 

In [8]:
#xpsim.export_simulation(geom, batch_stats, print_to_console=False, save_csv=True, output_dir=f'simulation_output_{geom.maj_xs_method}_{geom.maj_mat_method}_{geom.access_method}.csv')
xpcsv.export_cross_batch_stats(batch_stats, geom=geom, print_to_console= False, save_csv= True, output_dir=f'cross_batch_stats_{geom.maj_xs_method}_{geom.maj_mat_method}_{geom.access_method}.csv')


Cross-batch statistics saved → cross_batch_stats_group_simple_fly.csv


In [ ]:
print(slab1.nuclides)
print(slab1.total_density)

[(<openmc.data.multipole.WindowedMultipole object at 0x7d3514a68220>, 4.831863374901811e+22)]
4.831863374901811e+22


In [ ]:

# ============================================================================
# Materials
# ============================================================================
slab1 = Material(
    name     = "cell 1",
    nuclides = [('U238', N_U238)],
    T        = 293.6,    # K (~20 °C)
)

slab2 = Material(
    name     = "cell 2",
    nuclides = [('U238', N_U238)],
    T        = 2000,     # K
)

# ============================================================================
# Geometry  (two slabs)
# ============================================================================
geom = Geometry(majorant_log=True, verbose=False)
geom.xs_dir = '/home/paule/open_mc_projects/xs_lib/endfb-vii.1-hdf5/wmp'

geom.add_material(slab1)
geom.add_material(slab2)
geom.boundaries     = [0.0, 2.0, 15.0]  # cm
geom.material_array = [slab1, slab2]

# Flux tally: 5 energy groups
energy_bins = [10.0, 50.0, 600.0, 1e4, 1e6, 2e7]  # eV
geom.attach_flux_tally(energy_bins)

# Verification tally: same energy bins, surface at slab1/slab2 interface
geom.attach_verification_tally(
    energy_bins = energy_bins,
    surface_xs  = [2.0],        # slab1/slab2 interface
)

# Majorant and XS method
geom.set_mode("validation", filename='validation/xs_generation/statepoint.200.h5')
#geom.df_group_xs = df_cross_section

# ============================================================================
# Source  (monoenergetic, at x=0, pointing right)
# ============================================================================
N_NEUTRONS  = 100   # neutrons per batch
N_BATCHES   = 5    # number of independent batches (100 neutrons each)

src = Source(
    neutron_nbr  = N_NEUTRONS,
    energy_range = [500, 150],
    energy_dist  = "mono",
    position     = [0.0, 0.0, 0.0],
    direction    = [1.0, 0.0, 0.0],
)

print(geom.reconr_grid)



('U238', 4.831863374901811e+22)
('U238', 4.831863374901811e+22)
U238
U238
15.0
  Batch   1/5  (100 neutrons)  wall=1.036s
  Batch   2/5  (100 neutrons)  wall=1.142s
  Batch   3/5  (100 neutrons)  wall=0.976s
  Batch   4/5  (100 neutrons)  wall=1.003s
  Batch   5/5  (100 neutrons)  wall=0.991s

  CROSS-BATCH STATISTICS

  FLUX TALLY [cm · src-n⁻¹]
  Group                               Mean           ±Std    Rel.Err
  -----------------------------------------------------------------
  10-50 eV                      0.0000e+00     0.0000e+00        inf
  50-600 eV                     1.4127e+01     4.0099e-01     0.0284
  600-10000 eV                  0.0000e+00     0.0000e+00        inf
  10000-1000000 eV              0.0000e+00     0.0000e+00        inf
  1000000-20000000 eV           0.0000e+00     0.0000e+00        inf

  ABSORPTION RATE [reactions · src-n⁻¹]
  Region / Group                      Mean           ±Std    Rel.Err
  ---------------------------------------------------------

In [ ]:
# ============================================================================
# Run — batch mode
# ============================================================================
batch_stats = geom.run_batch(src, N_BATCHES)
xpcsv.export_cross_batch_stats(batch_stats, geom, print_to_console=True)
# ============================================================================
# Results
# ============================================================================
#print(geom.summary())


print(geom._nuclides_wmp_lib)
print(geom.nuclides)

{'U238': <openmc.data.multipole.WindowedMultipole object at 0x71305502adf0>}
{'U238': (<openmc.data.multipole.WindowedMultipole object at 0x71305502adf0>, 4.831863374901811e+22)}
